# BFCL-India: QLoRA Fine-Tuning Pipeline (v2)

This notebook contains the complete, self-contained pipeline for fine-tuning `Qwen2.5-3B-Instruct` on the optimized BFCL-India v2 training mix.

### Training Config:
- **Base model**: `Qwen/Qwen2.5-3B-Instruct`
- **Dataset**: balanced mix (50% xLAM unfiltered, 30% Indian-domain, 10% Glaive, 10% APIGen-MT)
- **Method**: LoRA ($r=16$, $\alpha=32$) via PEFT
- **Optimization**: learning rate $1e-4$ (halved), 1 epoch, batch size 2, gradient accumulation 4
- **Monitoring**: eval validation loss every 100 steps, save best model via `load_best_model_at_end=True` and early stopping.

In [ ]:
# Install dependencies
!pip install -q -U transformers peft trl datasets accelerate bitsandbytes

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import json
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    AutoTokenizer,
    TrainingArguments,
    EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

# Set seed
torch.manual_seed(42)

## 1. Load Training Data

In [ ]:
# Load dataset from local files if present, otherwise load directly from Hugging Face
if os.path.exists("data/train.jsonl"):
    dataset = load_dataset(
        "json",
        data_files={
            "train": "data/train.jsonl",
            "validation": "data/val.jsonl"
        }
    )
else:
    print("Local data not found. Loading directly from Hugging Face Hub...")
    dataset = load_dataset(
        "bhavjeetsingh2912/toolcaller-train-mix",
            verification_mode="no_checks",
        data_files={
            "train": "train.jsonl",
            "validation": "val.jsonl"
        }
    )

print(f"Train examples: {len(dataset['train'])}")
print(f"Val examples: {len(dataset['validation'])}")

## 2. Initialize Model & Tokenizer

In [ ]:
model_id = "Qwen/Qwen2.5-3B-Instruct"
from transformers import BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Configure 4-bit quantization for QLoRA efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load base model in 4-bit quantization
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map={"": 0},
    trust_remote_code=True
)
model.config.use_cache = False

## 3. Set Up LoRA Adapter

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 4. Training Arguments & SFTTrainer

In [ ]:
training_args = SFTConfig(
    output_dir="./outputs",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="steps",
    max_length=2048,
    save_strategy="steps",
    eval_steps=100,
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=True,
    optim="paged_adamw_32bit",
    gradient_checkpointing=True,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    args=training_args,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

## 5. Launch Training

In [ ]:
trainer.train()

## 6. Save Model Adapter

In [ ]:
model.save_pretrained("toolcaller-qwen-3b-v2-adapter")
print("Adapter saved successfully!")